# DCIM System - Demonstration & Redovisning
Denna notebook demonstrerar funktionaliteten i vårt objektorienterade DCIM-system (Datacenter Infrastructure Management).

### Innehåll:
1. Systeminitiering och inläsning från JSON-fröfil (`data/seed_devices.json`)
2. Demonstration av Objektorientering & Dunder-metoder (`__str__` och `__repr__`)
3. Polymorfism via systemdiagnostik (`kor_systemdiagnostik`)
4. Kapsling, validering och felhantering (Custom Exceptions & Properties)

In [1]:
import sys
from pathlib import Path

# Säkerställ att Python hittar paketet 'datacenter' när notebooken körs från mappen 'notebooks'
projektrot = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(projektrot) not in sys.path:
    sys.path.insert(0, str(projektrot))

from datacenter.manager.datacenter_manager import DatacenterManager
from datacenter.models.hardware import Server
from datacenter.models.connectivity import Switch
from datacenter.models.infra import Kylaggregat

print(" Alla moduler importerades framgångsrikt!")

 Alla moduler importerades framgångsrikt!


## 1. Initiering och inläsning från JSON-fröfil
Vi initierar `DatacenterManager` och läser in vår konfigurationsfil `data/seed_devices.json`.

In [2]:
# Skapa managern
manager = DatacenterManager()

# Ladda enheter från fröfilen
frofil_vag = projektrot / "data" / "seed_devices.json"
antal_inlasta = manager.ladda_fran_json(str(frofil_vag))

print(f" Läste in {antal_inlasta} enheter från {frofil_vag.name}\n")
print(f"Totalt antal enheter i systemet: {len(manager.enheter)}")

 Läste in 6 enheter från seed_devices.json

Totalt antal enheter i systemet: 6


## 2. Objektorientering & Dunder-metoder
Vi skapar instanser för hand och demonstrerar `__str__` (för användarutskrifter) samt `__repr__` (för tekniska listor och loggning).

In [3]:
# Skapa enskilda objekt
demo_server = Server("SRV-DEMO", "Dell", "10.0.0.5", "INTRANÄT")
demo_kyl = Kylaggregat("COOL-DEMO", "Rittal", temp=18.0)
demo_switch = Switch("SW-DEMO", "Cisco", antal_portar=24)

print("--- Användarvänlig utskrift via __str__ ---")
print(demo_server)
print(demo_kyl)
print(demo_switch)

print("\n--- Teknisk representation via __repr__ i en lista ---")
enhets_lista = [demo_server, demo_kyl, demo_switch]
print(enhets_lista)

--- Användarvänlig utskrift via __str__ ---
Server [SRV-DEMO] (Dell) - Status: OK
Kylaggregat [COOL-DEMO] (Rittal) - Status: OK
Switch [SW-DEMO] (Cisco) - Status: OK

--- Teknisk representation via __repr__ i en lista ---
[<Server(enhet_id='SRV-DEMO', fabrikat='Dell', status='OK')>, <Kylaggregat(enhet_id='COOL-DEMO', fabrikat='Rittal', status='OK')>, <Switch(enhet_id='SW-DEMO', fabrikat='Cisco', status='OK')>]


## 3. Polymorfism & Systemdiagnostik
Managern itererar över alla resurser i datacentret och anropar `kor_diagnostik()` dynamiskt, oavsett om det är en Server, ett Kylaggregat eller en Switch.

In [4]:
# Kör diagnostik på hela systemet
diagnostik_rapporter = manager.kor_systemdiagnostik()
print("\n".join(diagnostik_rapporter))

Serverdiagnostik OK för SRV-01 (192.168.1.10) på INTRANÄT.
Serverdiagnostik OK för SRV-02 (192.168.1.11) på DMZ.
Serverdiagnostik OK för SRV-03 (192.168.1.12) på DMZ.
Serverdiagnostik OK för SRV-04 (192.168.1.13) på INTRANÄT.
Kylanalys OK för COOL-01. Aktuell temp: 19.5°C.
Switchdiagnostik OK för SW-01. Alla 48 svarar.


## 4. Kapsling, Validering & Felhantering
Vi demonstrerar hur våra `@property`-setters stoppar ogiltig data (t.ex. negativt antal portar på en Switch) och kastar lämpliga undantag.

In [5]:
print("Försöker skapa en Switch med ogiltigt antal portar (-12)...")

try:
    ogiltig_switch = Switch("SW-FEL", "Cisco", antal_portar=-12)
except ValueError as e:
    print(f" Fångade förväntat fel: {e}")

print("\nÄndrar status på en befintlig server till CRITICAL:")
demo_server.status = "CRITICAL"
print(demo_server)

Försöker skapa en Switch med ogiltigt antal portar (-12)...
 Fångade förväntat fel: -12 är inte ett giltigt antal portar (måste vara ett positivt heltal).

Ändrar status på en befintlig server till CRITICAL:
Server [SRV-DEMO] (Dell) - Status: CRITICAL
